In [17]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
scaled_train = pd.read_csv('../data/processed/scaled_train.csv')

In [19]:
X, y = scaled_train.drop(columns=["result"]), scaled_train["result"]

In [20]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(1.2847038019451813), np.int64(1): np.float64(0.7888165038002172), np.int64(2): np.float64(1.0483405483405484)}


In [21]:
sample_weights = y.map(class_weights)

In [22]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Base models
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss"
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    # class_weight=class_weights
)

svc = SVC(
    probability=True,
    random_state=42,
    # class_weight=class_weights
)

# Ensemble
ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb),
        ("lr", lr),
        ("svc", svc)
    ],
    voting="soft"      # Uses probabilities
)

# Parameters to tune
param_grid = {
    "xgb__n_estimators": [50, 100],
    "xgb__learning_rate": [0.01, 0.05],
    "xgb__max_depth": [3, 5],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],

    "lr__C": [0.01, 0.05, 0.1],

    "svc__C": [0.01, 0.05, 0.1],
    "svc__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    ensemble,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y, sample_weight=sample_weights)

best_model = grid.best_estimator_

print("Best Parameters:")
print(grid.best_params_)

print("Best CV Accuracy:")
print(grid.best_score_)

# Predictions
y_pred = best_model.predict(X)

probs = best_model.predict_proba(X)

print(probs.shape)
print(probs[:5])

print("Train Accuracy:", accuracy_score(y, y_pred))


/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matche

Best Parameters:
{'lr__C': 0.01, 'svc__C': 0.01, 'svc__gamma': 'scale', 'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 3, 'xgb__n_estimators': 100, 'xgb__subsample': 1.0}
Best CV Accuracy:
0.48591268320216424
(1453, 3)
[[0.27335508 0.47659551 0.25004941]
 [0.28634232 0.37250569 0.34115198]
 [0.20658252 0.52722886 0.26618862]
 [0.32669296 0.32024551 0.35306155]
 [0.23884734 0.28564823 0.47550443]]
Train Accuracy: 0.6311080523055747


In [ ]:
# import joblib
# joblib.dump(best_model, "../models/best_model.pkl")

['../models/best_model.pkl']

In [26]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.24      0.36       377
           1       0.63      0.81      0.71       614
           2       0.61      0.71      0.66       462

    accuracy                           0.63      1453
   macro avg       0.66      0.59      0.58      1453
weighted avg       0.65      0.63      0.60      1453



# Inference